# 🏆 Lab 07 — Capstone: Production AI Assistant
**Build a complete, evaluated, cost-tracked AI assistant from scratch**

---
**Scenario:** You are a Data + AI Engineer at a Dutch financial institution.
Your task: build an internal knowledge assistant for 2,000 mortgage advisors who
currently waste 45 minutes/day searching policy documents manually.

**Architecture you will build:**
```
Documents → Chunking → Embeddings → ChromaDB (vector) + BM25 (keyword)
                                         ↓
User Query → Hybrid Retrieval → Re-ranking → LangGraph Agent → Answer
                                                   ↓
                           MLflow Traces ← Tool Calls → Tools
                                   ↓
                         RAGAS Eval + LLM Judge → Quality Dashboard
```

**Every component is production-grade:**
- MLflow traces every token, cost, and latency
- Hybrid retrieval + cross-encoder re-ranking
- LangGraph agent with tool use and human escalation
- RAGAS evaluation in a CI/CD-style gate
- Cost tracking and semantic caching

**Estimated time:** 90 min | **Level:** Advanced | **Databricks Free Edition + GPU recommended**

---
This lab integrates everything from Labs 01–06. Complete those first.

In [ ]:
%pip install -q openai chromadb sentence-transformers rank-bm25 langgraph langchain langchain-openai ragas datasets mlflow pandas numpy nltk

In [ ]:
import os, json, re, time, hashlib
import numpy as np
import pandas as pd
import nltk
import mlflow
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from openai import OpenAI
from datetime import datetime

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Models
embed_model  = SentenceTransformer('BAAI/bge-small-en-v1.5')
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
oai_client   = OpenAI()

# MLflow — built into Databricks, appears in your Experiments sidebar
mlflow.set_experiment('Production-Mortgage-Assistant')
mlflow.openai.autolog()   # every OpenAI call is automatically traced

print(f'Setup complete — {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Embed model: {embed_model.get_sentence_embedding_dimension()} dimensions')
print('MLflow autolog: active')

## Phase 1 — Document Ingestion and Smart Chunking

We use section-aware chunking for structured policy documents.
Each chunk carries metadata: section number, word count, creation date.
This metadata enables filtered retrieval (e.g., 'only search sections about LTV').

In [ ]:
# Three synthetic documents representing a real knowledge base
DOCUMENTS = [
    {
        'id': 'doc_mortgage_2025',
        'title': 'Mortgage Lending Policy 2025',
        'type': 'policy',
        'updated': '2025-01-15',
        'content': '''
1. INCOME REQUIREMENTS
Maximum mortgage is 4.5 times gross annual income. Joint applications: 100% of higher income
plus 70% of lower income. Self-employed: average of 3 years accounts, or lowest year if most recent.

2. LOAN-TO-VALUE (LTV)
Primary residence: maximum 100% of market value. Second property: maximum 90%.
NHG guarantee available for mortgages up to EUR 435,000.

3. STRESS TEST
Affordability calculated at 5.0% over 30-year term. Monthly obligations may not exceed
35% of gross monthly income. Car leases and student loans included. Childcare excluded.

4. ELIGIBLE PROPERTIES
Owner-occupied residential in Netherlands and Belgium. No commercial, agricultural, or foreign.
Leasehold (erfpacht) eligible if remaining term exceeds mortgage term by 10 years.

5. DOCUMENTATION
Required: payslip within 3 months, werkgeversverklaring, jaaropgave, property valuation, valid ID.

6. INTEREST RATE OPTIONS
Fixed: 1, 5, 10, 15, 20, 25, 30 years. Variable rate tracks ECB base rate.
Rate lock available up to 3 months before completion.
Penalty-free overpayment: up to 10% annually.

7. FIRST-TIME BUYERS
Transfer tax exemption (overdrachtsbelasting) on properties up to EUR 510,000.
Starter Loan (Starterslening) available as supplement.
'''
    },
    {
        'id': 'doc_kyc_2025',
        'title': 'KYC & Account Opening Guidelines 2025',
        'type': 'compliance',
        'updated': '2025-03-01',
        'content': '''
1. INDIVIDUAL ACCOUNT OPENING
Required documents: valid passport or EU identity card, proof of address (utility bill or bank
statement within 90 days), BSN number (Dutch citizen) or tax identification number.

2. BUSINESS ACCOUNT OPENING
Required: KvK (Chamber of Commerce) extract maximum 3 months old, valid ID of all directors,
proof of address for the business, UBO declaration for beneficial owners above 25%.

3. ENHANCED DUE DILIGENCE
Required for: PEPs (politically exposed persons), high-risk countries, transactions above EUR 15,000
in cash, non-resident accounts. Additional documentation: source of funds declaration,
business activity description, bank references.

4. ONGOING MONITORING
Annual review for high-risk customers. Triennial review for standard customers.
Automatic flag: transactions inconsistent with stated business purpose.
STR (suspicious transaction report) to FIU-NL within 24 hours of suspicion.

5. SANCTIONS SCREENING
All new customers screened against EU, UN, OFAC, and Dutch sanctions lists at onboarding.
Ongoing transaction monitoring against updated sanctions lists daily.
'''
    },
    {
        'id': 'doc_products_2025',
        'title': 'Product Guide: Savings and Investment 2025',
        'type': 'product',
        'updated': '2025-02-10',
        'content': '''
1. SAVINGS ACCOUNTS
Oranje Spaarrekening: variable rate, no minimum balance, instant access.
Rental: fixed-term 1-5 years, higher rate, 3-month notice for early withdrawal.
Deposit Guarantee Scheme (DGS) covers up to EUR 100,000 per person per institution.

2. INVESTMENT PRODUCTS
Zelf Beleggen: self-directed via ING app. Transaction cost: EUR 7.50 flat.
Beheerd Beleggen: managed portfolio, three risk profiles (conservative/balanced/growth).
Minimum investment: EUR 1,000. Annual management fee: 0.99% (max EUR 750).

3. PENSION PRODUCTS
ING Pensioenrekening: tax-deductible contributions up to annual limit.
Self-employed pension contribution limit: 30% of net profit, max EUR 38,049 (2025).
Funds locked until age 67, with exceptions for disability or business cessation.

4. SUSTAINABLE INVESTING
ESG filter available for all managed portfolios. Excludes companies with >5% revenue
from tobacco, weapons, fossil fuel extraction. SFDR Article 8 and Article 9 funds available.
'''
    }
]

def section_chunk_with_metadata(doc: dict) -> list:
    sections = re.split(r'(?=\n\d+\.\s+[A-Z])', doc['content'])
    chunks = []
    for i, section in enumerate(sections):
        text = section.strip()
        if len(text) < 50:
            continue
        section_match = re.match(r'(\d+)\.\s+([A-Z][A-Z\s&()]+)', text)
        section_num   = section_match.group(1) if section_match else str(i)
        section_title = section_match.group(2).strip() if section_match else 'Unknown'
        chunks.append({
            'id':            f'{doc["id"]}_s{section_num}',
            'text':          text,
            'doc_id':        doc['id'],
            'doc_title':     doc['title'],
            'doc_type':      doc['type'],
            'section':       section_title,
            'section_num':   int(section_num) if section_num.isdigit() else 0,
            'word_count':    len(text.split()),
            'updated':       doc['updated'],
        })
    return chunks

all_chunks = []
for doc in DOCUMENTS:
    all_chunks.extend(section_chunk_with_metadata(doc))

print(f'Total chunks: {len(all_chunks)}')
for chunk in all_chunks[:3]:
    print(f'  [{chunk["id"]}] {chunk["section"]} ({chunk["word_count"]} words)')

## Phase 2 — Vector Store and Hybrid Retrieval

In [ ]:
# Build ChromaDB + BM25 index
chroma = chromadb.Client()
collection = chroma.create_collection('mortgage_assistant', get_or_create=True)

texts      = [c['text'] for c in all_chunks]
embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

collection.add(
    ids        = [c['id'] for c in all_chunks],
    documents  = texts,
    embeddings = embeddings.tolist(),
    metadatas  = [{k: v for k, v in c.items() if k != 'text'} for c in all_chunks]
)

bm25_corpus = [t.lower().split() for t in texts]
bm25_index  = BM25Okapi(bm25_corpus)

print(f'Indexed {collection.count()} chunks in ChromaDB + BM25')

# Semantic cache: stores (query_hash → answer) to avoid duplicate LLM calls
semantic_cache: dict = {}

def get_cached_answer(query: str, threshold: float = 0.92) -> str | None:
    q_emb = embed_model.encode(query, normalize_embeddings=True)
    for cached_q, cached_data in semantic_cache.items():
        sim = float(np.dot(cached_data['embedding'], q_emb))
        if sim >= threshold:
            return cached_data['answer']
    return None

def set_cache(query: str, answer: str):
    q_emb = embed_model.encode(query, normalize_embeddings=True)
    semantic_cache[query] = {'embedding': q_emb, 'answer': answer}

def hybrid_retrieve(query: str, k: int = 5, doc_type: str | None = None) -> list:
    q_emb = embed_model.encode(query, normalize_embeddings=True)

    # Dense retrieval
    where_filter = {'doc_type': {'$eq': doc_type}} if doc_type else None
    dense_res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=min(10, collection.count()),
        where=where_filter
    )
    dense_docs = list(zip(dense_res['documents'][0], dense_res['metadatas'][0]))

    # Sparse retrieval (BM25)
    bm25_scores = bm25_index.get_scores(query.lower().split())
    sparse_top  = np.argsort(bm25_scores)[::-1][:10]
    sparse_docs = [(texts[i], all_chunks[i]) for i in sparse_top if bm25_scores[i] > 0]

    # Reciprocal Rank Fusion
    rrf_scores: dict = {}
    doc_lookup: dict = {}
    for rank, (text, meta) in enumerate(dense_docs):
        key = text[:80]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (60 + rank + 1)
        doc_lookup[key] = (text, meta)
    for rank, (text, meta) in enumerate(sparse_docs):
        key = text[:80]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (60 + rank + 1)
        doc_lookup[key] = (text, meta)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [doc_lookup[key] for key, _ in ranked]

def rerank(query: str, docs: list, top_k: int = 3) -> list:
    if not docs:
        return []
    texts_only = [d[0] for d in docs]
    scores = rerank_model.predict([(query, t) for t in texts_only])
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

# Test retrieval
q = 'What is the maximum LTV ratio for buying a second property?'
candidates = hybrid_retrieve(q, k=5)
top_chunks  = rerank(q, candidates, top_k=2)
print(f'\nQuery: "{q}"')
print(f'Retrieved {len(candidates)} candidates, re-ranked to top {len(top_chunks)}')
print(f'Best chunk: [{top_chunks[0][1]["section"]}] {top_chunks[0][0][:120]}...')

## Phase 3 — LangGraph Agent with Tools and Human Escalation

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, List
import operator

class AgentState(TypedDict):
    question:       str
    retrieved_docs: list
    answer:         str
    confidence:     float
    needs_human:    bool
    tokens_used:    int
    cache_hit:      bool

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

SYSTEM_PROMPT = '''You are an expert mortgage advisor assistant.
Answer questions ONLY using the provided context.
If the context does not contain the answer, say exactly: "I need to escalate this to a human advisor."
Be precise with numbers, percentages, and legal terms.
Include the relevant policy section reference in your answer.'''

def retrieve_node(state: AgentState) -> AgentState:
    '''Retrieve and re-rank relevant chunks'''
    candidates  = hybrid_retrieve(state['question'], k=6)
    top_chunks  = rerank(state['question'], candidates, top_k=3)
    return {**state, 'retrieved_docs': top_chunks}

def generate_node(state: AgentState) -> AgentState:
    '''Generate answer from retrieved context'''
    context = '\n\n'.join(
        f'[{meta["doc_title"]} — {meta["section"]}]\n{text}'
        for text, meta in state['retrieved_docs']
    )
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f'Context:\n{context}\n\nQuestion: {state["question"]}')
    ]
    response = llm.invoke(messages)
    answer   = response.content

    # Simple confidence heuristic: check if answer contains key indicators of grounding
    confidence = 0.5
    if any(kw in answer.lower() for kw in ['%', 'eur', 'section', 'policy', 'required', 'maximum']):
        confidence = 0.85
    if 'escalate' in answer.lower() or 'cannot find' in answer.lower():
        confidence = 0.1

    return {**state, 'answer': answer, 'confidence': confidence}

def escalation_check(state: AgentState) -> str:
    '''Decide: return answer or escalate to human'''
    if state['confidence'] < 0.3 or 'escalate' in state['answer'].lower():
        return 'escalate'
    return 'respond'

def escalate_node(state: AgentState) -> AgentState:
    return {**state,
            'needs_human': True,
            'answer': f'This question requires a human advisor. Reason: low confidence ({state["confidence"]:.0%}). Original draft: {state["answer"][:200]}'}

# Build LangGraph
graph = StateGraph(AgentState)
graph.add_node('retrieve', retrieve_node)
graph.add_node('generate', generate_node)
graph.add_node('escalate', escalate_node)
graph.set_entry_point('retrieve')
graph.add_edge('retrieve', 'generate')
graph.add_conditional_edges('generate', escalation_check, {'respond': END, 'escalate': 'escalate'})
graph.add_edge('escalate', END)
agent = graph.compile()

print('LangGraph agent compiled. Nodes: retrieve → generate → [respond | escalate]')

## Phase 4 — Full Pipeline with MLflow Tracing

In [ ]:
MODEL_COST_PER_TOKEN = 0.60 / 1_000_000  # gpt-4o-mini output: $0.60/1M tokens

@mlflow.trace(name='mortgage_assistant_query', span_type='CHAIN')
def answer_question(question: str, session_id: str = 'session_default') -> dict:
    t0 = time.time()

    # Check semantic cache first
    cached = get_cached_answer(question)
    if cached:
        return {
            'question':   question,
            'answer':     cached,
            'cache_hit':  True,
            'latency_ms': round((time.time() - t0) * 1000),
            'cost_usd':   0.0,
            'needs_human': False,
        }

    # Run agent
    initial_state: AgentState = {
        'question':       question,
        'retrieved_docs': [],
        'answer':         '',
        'confidence':     0.0,
        'needs_human':    False,
        'tokens_used':    0,
        'cache_hit':      False,
    }
    final_state = agent.invoke(initial_state)
    latency_ms  = round((time.time() - t0) * 1000)

    # Estimate cost (rough: 600 input tokens + 150 output tokens per query)
    est_cost = 750 * MODEL_COST_PER_TOKEN

    # Cache the answer
    if not final_state['needs_human']:
        set_cache(question, final_state['answer'])

    return {
        'question':     question,
        'answer':       final_state['answer'],
        'confidence':   final_state['confidence'],
        'needs_human':  final_state['needs_human'],
        'num_chunks':   len(final_state['retrieved_docs']),
        'sources':      [m['section'] for _, m in final_state['retrieved_docs']],
        'cache_hit':    False,
        'latency_ms':   latency_ms,
        'cost_usd':     est_cost,
    }

# Test queries
test_questions = [
    'I earn 90,000 euros gross annually. What is the maximum mortgage I qualify for?',
    'What documents do I need to open a business account?',
    'Can I invest in ESG-screened funds through ING?',
    'What is the maximum LTV for buying a second property?',
    'I earn 90,000 euros gross annually. What is the maximum mortgage I qualify for?',  # duplicate — tests cache
]

results = []
with mlflow.start_run(run_name=f'prod-test-{datetime.now().strftime("%Y%m%d-%H%M")}'):
    for i, q in enumerate(test_questions):
        result = answer_question(q, session_id=f'test_session_{i}')
        results.append(result)

        status = '[CACHE HIT]' if result['cache_hit'] else f'[{result["latency_ms"]}ms, ${result["cost_usd"]:.5f}]'
        print(f'Q{i+1}: {q[:60]}...')
        print(f'     A: {result["answer"][:120]}...')
        print(f'     {status} | confidence={result.get("confidence", 0):.0%} | sources={result.get("sources", [])}\n')

    # Log aggregate metrics
    non_cached = [r for r in results if not r['cache_hit']]
    mlflow.log_metrics({
        'total_queries':      len(results),
        'cache_hit_rate':     sum(r['cache_hit'] for r in results) / len(results),
        'avg_latency_ms':     sum(r['latency_ms'] for r in results) / len(results),
        'total_cost_usd':     sum(r['cost_usd'] for r in results),
        'escalation_rate':    sum(r['needs_human'] for r in results) / len(results),
    })

## Phase 5 — RAGAS Evaluation with CI/CD Gate

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

EVAL_SET = [
    {
        'question': 'What is the maximum LTV for a primary residence?',
        'ground_truth': 'The maximum LTV for a primary owner-occupied residence is 100% of the market value as assessed by a certified appraiser.'
    },
    {
        'question': 'What is the stress test interest rate at ING?',
        'ground_truth': 'ING applies a standard stress test interest rate of 5.0% over a 30-year term.'
    },
    {
        'question': 'What is the NHG guarantee threshold in 2025?',
        'ground_truth': 'The National Mortgage Guarantee (NHG) is available for mortgages up to EUR 435,000 in 2025.'
    },
    {
        'question': 'How much can I overpay annually without penalty?',
        'ground_truth': 'You can make penalty-free overpayments of up to 10% of the original loan amount per year.'
    },
]

# Run evaluation queries through the agent
for item in EVAL_SET:
    r = answer_question(item['question'])
    item['answer']   = r['answer']
    item['contexts'] = [text for text, _ in r.get('_retrieved_docs', [])] or [
        t for t, _ in hybrid_retrieve(item['question'], k=3)
    ]

eval_dataset = Dataset.from_list(EVAL_SET)

QUALITY_THRESHOLDS = {
    'faithfulness':     0.80,
    'answer_relevancy': 0.75,
    'context_precision': 0.65,
}

with mlflow.start_run(run_name='ragas-eval-gate'):
    scores = evaluate(eval_dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    mean_scores = scores.to_pandas().mean(numeric_only=True).to_dict()

    for metric, score in mean_scores.items():
        if isinstance(score, float):
            mlflow.log_metric(metric, score)

    print('\n── RAGAS Evaluation Results ──────────────────────')
    all_pass = True
    for metric, threshold in QUALITY_THRESHOLDS.items():
        score = mean_scores.get(metric, 0)
        if not isinstance(score, float):
            continue
        passed = score >= threshold
        status = '✓ PASS' if passed else '✗ FAIL'
        if not passed:
            all_pass = False
        print(f'  {metric:<30} {score:.3f}  (threshold: {threshold})  {status}')

    print('──────────────────────────────────────────────────')
    if all_pass:
        print('  ✅ All metrics passed — DEPLOYMENT APPROVED')
    else:
        print('  ❌ One or more metrics below threshold — DEPLOYMENT BLOCKED')
        print('  In CI/CD: this cell would call sys.exit(1) to fail the pipeline')

## Phase 6 — Production Dashboard (Cost, Quality, Escalation)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Simulate 30 days of production metrics
np.random.seed(42)
days = list(range(1, 31))
daily_queries   = np.random.randint(800, 1400, 30)
cache_rates     = np.random.uniform(0.28, 0.45, 30)
faithfulness    = 0.82 + np.cumsum(np.random.normal(0.002, 0.008, 30))  # slowly improving
faithfulness    = np.clip(faithfulness, 0.7, 0.98)
escalation_rate = np.random.uniform(0.04, 0.12, 30)
avg_latency_ms  = np.random.uniform(650, 1200, 30) * (1 - cache_rates * 0.6)
daily_cost_usd  = daily_queries * (1 - cache_rates) * 750 * MODEL_COST_PER_TOKEN

fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor('#060D1A')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

panels = [
    (0, 0, daily_queries,   '#00B4D8', 'Daily Queries',       'Queries'),
    (0, 1, cache_rates*100, '#9B5DE5', 'Cache Hit Rate',      '%'),
    (0, 2, faithfulness,    '#E8A020', 'RAGAS Faithfulness',  'Score'),
    (1, 0, escalation_rate*100, '#E05A4E', 'Escalation Rate (%)', '%'),
    (1, 1, avg_latency_ms,  '#2EC4B6', 'Avg Latency',         'ms'),
    (1, 2, daily_cost_usd,  '#F77F00', 'Daily API Cost',      'USD'),
]

for row, col, data, color, title, ylabel in panels:
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#0A1628')
    ax.plot(days, data, color=color, linewidth=2)
    ax.fill_between(days, data, alpha=0.12, color=color)
    ax.set_title(title, color='#E8E4DC', fontsize=10, pad=8)
    ax.set_ylabel(ylabel, color='#6B7E99', fontsize=8)
    ax.tick_params(colors='#4A5568', labelsize=7)
    for spine in ax.spines.values():
        spine.set_color('#1A2840')
    # Add threshold line for quality metrics
    if title == 'RAGAS Faithfulness':
        ax.axhline(0.80, color='#E05A4E', linestyle='--', linewidth=1, alpha=0.6, label='Threshold (0.80)')
        ax.legend(fontsize=7, labelcolor='white', framealpha=0.1)

fig.suptitle('Mortgage Assistant — 30-Day Production Dashboard', color='#F0EBE0', fontsize=14, y=0.98)
plt.show()

# Summary statistics
print('\n── 30-Day Summary ────────────────────────────────────')
print(f'  Total queries served:     {daily_queries.sum():,}')
print(f'  Avg cache hit rate:       {cache_rates.mean():.1%}')
print(f'  Total API cost:           ${daily_cost_usd.sum():.2f}')
print(f'  Avg RAGAS faithfulness:   {faithfulness.mean():.3f}')
print(f'  Avg escalation rate:      {escalation_rate.mean():.1%}')
print(f'  Avg response latency:     {avg_latency_ms.mean():.0f}ms')
print(f'  Cost per query:           ${daily_cost_usd.sum() / daily_queries.sum():.5f}')

## 🏆 Capstone Complete

You have built a complete production AI assistant:

| Component | Implementation |
|-----------|---------------|
| Document ingestion | Section-aware chunking with metadata |
| Vector store | ChromaDB with sentence-transformers embeddings |
| Retrieval | Hybrid BM25 + dense + Reciprocal Rank Fusion |
| Re-ranking | Cross-encoder (ms-marco-MiniLM) |
| Agent | LangGraph: retrieve → generate → [respond|escalate] |
| Tracing | MLflow autolog + custom spans — every token traced |
| Caching | Semantic cache — duplicate queries at zero cost |
| Evaluation | RAGAS (faithfulness, relevancy, precision) |
| Quality gate | CI/CD eval gate — blocks deploy on regression |
| Dashboard | 6-panel MLflow-tracked production metrics |

**This is your portfolio project.** Add your own documents, deploy to Databricks Model Serving,
connect a Streamlit or Gradio frontend, and you have a production-ready AI system.

---

### Next Steps for Production Deployment

1. **Replace ChromaDB** with Weaviate Cloud (free sandbox) or Databricks Vector Search for scalability
2. **Add LiteLLM gateway** in front of the OpenAI calls for routing, fallbacks, and cost control
3. **Deploy as Databricks Model Serving endpoint** — one click, auto-scales, monitoring built in
4. **Set up Langfuse or MLflow alerts** — get Slack notification when faithfulness drops below 0.80
5. **Add fine-tuning pipeline** (Lab 06) when prompt + RAG can no longer close quality gaps
6. **A/B test prompt versions** — route 10% of traffic to new prompt, compare RAGAS scores

**GitHub it.** A working capstone with MLflow traces, RAGAS scores, and a production dashboard
is worth more in a Data + AI Engineer interview than any certification.